In [1]:
from src.haive.agents.react_agent.agent import ReactAgent,ReactAgentConfig

USER_AGENT environment variable not set, consider setting it to identify your requests.
/home/will/Projects/haive/backend/haive/.venv/lib/python3.12/site-packages/pydantic/_internal/_config.py:345: UserWarning: Valid config keys have changed in V2:
* 'allow_population_by_field_name' has been renamed to 'populate_by_name'
* 'orm_mode' has been renamed to 'from_attributes'
  warnings.warn(message, UserWarning)


Setting system prompt to 'You are a helpful assistant.'


ImportError: cannot import name 'AgentArchitectureConfig' from 'src.haive.agents.base' (/home/will/Projects/haive/backend/haive/src/haive/agents/base.py)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from typing import Any, Dict

from collections import defaultdict
from typing import Any, Dict, Tuple


class StateGraphManager:
    """
    A manager for extracting metadata and modifying a StateGraph.
    """

    def __init__(self, graph: Any):
        """
        Initialize the StateGraphManager.

        Args:
            graph (StateGraph): The StateGraph object to manage.
        """
        self.graph = graph
        self.metadata = self.extract_metadata()
        self.needs_recompile = False  # Track modifications requiring recompilation

    def extract_metadata(self) -> Dict[str, Any]:
        """
        Extract metadata from the StateGraph, including conditional branches.
        """
        metadata = {
            "entry_point": getattr(self.graph, "entry_point", None),
            "finish_point": getattr(self.graph, "finish_point", None),
            "nodes": list(self.graph.nodes.keys()),
            "edges": list(self.graph.edges),
            "conditional_edges": defaultdict(list),
            "schemas": getattr(self.graph, "schemas", {}),
            "input_schema": getattr(self.graph, "input", None),
            "output_schema": getattr(self.graph, "output", None),
            "compiled": getattr(self.graph, "compiled", False),
            "support_multiple_edges": getattr(self.graph, "support_multiple_edges", True),
        }

        branches = getattr(self.graph, "branches", {})
        for node, conditions in branches.items():
            for condition_name, branch_obj in conditions.items():
                if hasattr(branch_obj, "ends"):
                    for condition, target in branch_obj.ends.items():
                        metadata["conditional_edges"][node].append(
                            (condition, "END" if target == "__end__" else target)
                        )

        return metadata

    def ensure_compiled(self):
        """Recompile the graph if modifications were made."""
        if self.needs_recompile:
            print("🔄 Recompiling the graph...")
            self.graph.compile()
            self.metadata = self.extract_metadata()
            self.needs_recompile = False

    def add_node(self, node: str):
        """Add a node to the graph."""
        if node not in self.graph.nodes:
            self.graph.nodes[node] = {}
            self.needs_recompile = True

    def remove_edge(self, src: str, dst: str):
        """Remove an edge from the graph."""
        if (src, dst) in self.graph.edges:
            self.graph.edges.remove((src, dst))
            self.needs_recompile = True

    #def insert_node(self, node: str, between: Tuple[str, str], func: callable = None):
    def insert_node(self, node: str, between: Tuple[str, str], func: callable = None):
        """
        Insert a new node between two existing nodes, using LangGraph's `add_node` and `add_edge` methods.
    
        Args:
            node (str): The name of the new node.
            between (Tuple[str, str]): A tuple (src, dst) indicating where to insert the node.
            func (callable, optional): The function that the node represents in LangGraph.
        """
        src, dst = between
    
        if src not in self.graph.nodes or dst not in self.graph.nodes:
            self.graph.add_node(node,func)
            raise ValueError(f"Cannot insert node: {src} or {dst} does not exist in the graph.")
    
        print(f"📌 Inserting `{node}` between `{src}` → `{dst}`")
    
        # Remove the existing edge
        self.remove_edge(src, dst)
    
        # If function is not provided, try to extract from the existing graph
        if func is None:
            func = getattr(self.graph, node, None)  # Try extracting from existing graph
            if not callable(func):
                raise ValueError(f"No callable function found for `{node}`. Provide `func` explicitly.")
    
        # ✅ Add the new node with its function in LangGraph
        self.graph.add_node(func, node)
    
        # ✅ Add new edges
        self.graph.add_edge(src, node)
        self.graph.add_edge(node, dst)
    
        self.needs_recompile = True
    





    def insert_start_node(self, node: str):
        """Insert a node into the branch between `__start__` and the first connected node."""
    
        # Retrieve edges originating from `__start__`
        start_edges = [edge for edge in self.graph.edges if edge[0] == "__start__"]
    
        if not start_edges:
            raise ValueError("No existing start edges found. Ensure `__start__` is connected in the graph.")
    
        # Pick the first transition from `__start__`
        _, first_node = start_edges[0]
    
        print(f"📌 Inserting {node} between __start__ → {first_node}")
    
        # Add the new node
        self.add_node(node)
    
        # Remove old edge and insert new ones
        self.remove_edge("__start__", first_node)
        self.graph.edges.add(("__start__", node))
        self.graph.edges.add((node, first_node))
    
        self.needs_recompile = True
    
    

    def insert_end_node(self, node: str):
        """Insert a node into the branch before `END`."""
    
        # Retrieve edges that connect to `END`
        end_edges = [edge for edge in self.graph.edges if edge[1] == "END"]
    
        if not end_edges:
            raise ValueError("No existing end edges found. Ensure `END` is connected in the graph.")
    
        # Pick the first transition to `END`
        last_node, _ = end_edges[0]
    
        print(f"📌 Inserting {node} between {last_node} → END")
    
        # Add the new node
        self.add_node(node)
    
        # Remove old edge and insert new edges
        #self.remove_edge(last_node, "END")
        #self.graph.edges.add((last_node, node))
        #self.graph.edges.add((node, "END"))

        #self.needs_recompile = True



    def update_branch(self, node: str, condition: str, target: str):
        """Update a conditional branch using defaultdict."""
        if "branches" not in self.graph.__dict__:
            self.graph.branches = defaultdict(dict)
        if node not in self.graph.branches:
            self.graph.branches[node] = {}

        self.graph.branches[node][condition] = target
        self.needs_recompile = True

    def get_metadata(self):
        """Return the extracted metadata."""
        return self.metadata

    def __del__(self):
        """Ensure compilation before object is deleted."""
        self.ensure_compiled()

    def visualize(self, output_file: str = "state_graph.png"):
        """
        Visualize the StateGraph using NetworkX, ensuring **arrows are drawn correctly**.

        Args:
            output_file (str): The filename to save the visualization.
        """
        G = nx.DiGraph()
        solid_edges = []
        dashed_edges = []
        edge_labels = {}

        # ✅ Add nodes
        for node in self.metadata["nodes"]:
            G.add_node(node)

        # ✅ Add standard edges (Solid)
        for src, dst in self.metadata["edges"]:
            if dst == "__end__":
                dst = "END"
            G.add_edge(src, dst)
            solid_edges.append((src, dst))

        # ✅ Add conditional branching edges (Dashed)
        for node, conditions in self.metadata["conditional_edges"].items():
            for condition, target in conditions:
                G.add_edge(node, target)
                dashed_edges.append((node, target))
                edge_labels[(node, target)] = f"{condition}"

        # ✅ Layout for better separation
        plt.figure(figsize=(14, 8))
        pos = nx.spring_layout(G, seed=42)  # More natural positioning

        # **Draw Solid Edges (State Transitions)**
        nx.draw_networkx_edges(
            G, pos, edgelist=solid_edges,
            edge_color="black", width=2, alpha=0.8,
            arrows=True, arrowstyle="-|>", arrowsize=20,  # 🔥 Force arrows
            connectionstyle="arc3,rad=0.1"
        )

        # **Draw Dashed Conditional Edges (Branching Paths)**
        nx.draw_networkx_edges(
            G, pos, edgelist=dashed_edges,
            edge_color="red", style="dashed", width=2,
            arrows=True, arrowstyle="-|>", arrowsize=20,  # 🔥 Force arrows
            connectionstyle="arc3,rad=0.3"
        )

        # **Draw Nodes**
        nx.draw_networkx_nodes(G, pos, node_color="lightblue", node_size=2800, edgecolors="black")
        nx.draw_networkx_labels(G, pos, font_size=12, font_weight="bold")

        # **Draw Conditional Labels**
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color="red", font_size=10)

        # ✅ Highlight entry and finish points
        entry = self.metadata["entry_point"]
        finish = self.metadata["finish_point"]

        if entry:
            nx.draw_networkx_nodes(G, pos, nodelist=[entry], node_color="green", node_size=3000, edgecolors="black")  # Entry point
        if finish and finish in self.metadata["nodes"]:
            nx.draw_networkx_nodes(G, pos, nodelist=[finish], node_color="red", node_size=3000, edgecolors="black")  # Finish point

        # ✅ Save and display
        plt.title("State Graph Visualization", fontsize=14, fontweight="bold")
        plt.savefig(output_file, bbox_inches="tight")
        plt.show()
        print(f"Graph saved as {output_file}")

    def get_metadata(self):
        """Return the extracted metadata."""
        return self.metadata


In [ ]:
a = ReactAgentConfig(should_compile=False)

In [ ]:
agent = ReactAgent(a)

In [ ]:
print(agent.graph)

In [ ]:
graph2 = agent.graph

In [ ]:
agent.compile_workflow()

In [ ]:
graph_manager = StateGraphManager(agent.graph)

# Example function for the new node
def process_state(state):
    return {"messages": ["Processing..."]}

# Insert a node between 'tool_node' and 'agent_node', adding the function
graph_manager.insert_node("sta", ("tool_node", "agent_node"), func=process_state)

# Debug: Print after modifications
print("✅ Updated Graph:", graph_manager.get_metadata())

# Recompile if needed
graph_manager.ensure_compiled()


In [ ]:
# Keys

In [2]:
agent.graph.branches['agent_node']

NameError: name 'agent' is not defined

In [3]:
agent.graph.nodes

NameError: name 'agent' is not defined

In [4]:
agent.graph.nodes.keys()
print(dir(agent.graph))

NameError: name 'agent' is not defined

In [5]:
print(agent.graph.config_schema)

NameError: name 'agent' is not defined

In [6]:
print(agent.graph.input)

NameError: name 'agent' is not defined

In [7]:
#print(dir(agent.app.map()))
def sta(state):
    return {'messages':['hi']}
    

In [8]:
print(agent.graph._all_edges)
agent.graph.add_node(sta,'sta')

NameError: name 'agent' is not defined

In [9]:
agent.graph.add_edge('tool_node','sta')

NameError: name 'agent' is not defined

In [10]:
agent.graph.add_edge('sta','agent_node')

NameError: name 'agent' is not defined

In [11]:
print(agent.graph.branches)

NameError: name 'agent' is not defined

In [12]:
print(agent.graph.edges)
agent.graph.edges.discard(('tool_node', 'agent_node'))
agent.graph.edges.add(('sta','agent_node'))

NameError: name 'agent' is not defined

In [13]:
agent.graph.compile()

NameError: name 'agent' is not defined

In [14]:
graph_manager = StateGraphManager(agent.graph)

# Debug: Print before modifications
print("🔍 Before Modifications:", graph_manager.get_metadata())

# Insert a node between 'tool_node' and 'agent_node'
graph_manager.insert_node("new_mid_node", ("tool_node", "agent_node"))

# Insert a node into the START branch
#graph_manager.insert_start_node("start_mid_node")

# Insert a node into the END branch
#graph_manager.insert_end_node("final_check_node")

# Debug: Print after modifications
print("✅ After Modifications:", graph_manager.get_metadata())

# Force recompilation only when needed
#graph_manager.graph.compile()
agent.graph.compile()

NameError: name 'StateGraphManager' is not defined

In [15]:
#print(dir(agent.app.map()))
def sta(state):
    return {'messages':['hi']}
    
#agent.graph.add_node(sta,'sta')
agent.graph.add_node(sta,'sta')
agent.graph.add_edge('sta','agent_node')


NameError: name 'agent' is not defined

In [16]:
from IPython.display import Image, display

try:
    display(Image(agent.app.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [17]:
print(agent.graph.edges)
agent.graph.compile()
print(dir(agent.app))
print(agent.app.transform)
print(dir(agent.graph))

NameError: name 'agent' is not defined

In [18]:

graph_manager = StateGraphManager(agent.graph)

# Insert a node between 'tool_node' and 'agent_node'
graph_manager.insert_node("new_mid_node", ("tool_node", "agent_node"))

# Insert a node right after the start node
graph_manager.insert_start_node("start_mid_node")

# Insert a node right before the END node
graph_manager.insert_end_node("final_check_node")

# Force recompilation only when needed
graph_manager.ensure_compiled()


NameError: name 'StateGraphManager' is not defined

In [19]:
print(graph2.compiled)

NameError: name 'graph2' is not defined

In [20]:
agent.graph.remove_node('agent_node')

NameError: name 'agent' is not defined

In [21]:
import networkx as nx
import matplotlib.pyplot as plt
from typing import Any, Dict


class StateGraphManager:
    """
    A manager for extracting metadata and visualizing a StateGraph with better aesthetics.
    """

    def __init__(self, graph: Any):
        """
        Initialize the StateGraphManager.

        Args:
            graph (StateGraph): The StateGraph object to manage.
        """
        self.graph = graph
        self.metadata = self.extract_metadata()

    def extract_metadata(self) -> Dict[str, Any]:
        """
        Extract metadata from the StateGraph, including conditional branches.

        Returns:
            Dict[str, Any]: A dictionary containing metadata.
        """
        metadata = {
            "entry_point": getattr(self.graph, "entry_point", None),
            "finish_point": getattr(self.graph, "finish_point", None),
            "nodes": list(getattr(self.graph, "nodes", {}).keys()),
            "edges": list(getattr(self.graph, "edges", set())),
            "conditional_edges": {},  # Extracted below
            "schema":getattr(self.graph,'schema',{}),
            "schemas": getattr(self.graph, "schemas", {}),
            "input_schema": getattr(self.graph, "input", None),
            "output_schema": getattr(self.graph, "output", None),
            "compiled": getattr(self.graph, "compiled", False),
            "support_multiple_edges": getattr(self.graph, "support_multiple_edges", True),
        }

        # ✅ Extract Conditional Edges
        branches = getattr(self.graph, "branches", {})
        for node, conditions in branches.items():
            for condition_name, branch_obj in conditions.items():
                if hasattr(branch_obj, "ends"):
                    for condition, target in branch_obj.ends.items():
                        if node not in metadata["conditional_edges"]:
                            metadata["conditional_edges"][node] = []
                        metadata["conditional_edges"][node].append(
                            (condition, "END" if target == "__end__" else target)
                        )

        return metadata

    def visualize(self, output_file: str = "state_graph.png"):
        """
        Visualize the StateGraph using NetworkX, ensuring **arrows are drawn correctly**.

        Args:
            output_file (str): The filename to save the visualization.
        """
        G = nx.DiGraph()
        solid_edges = []
        dashed_edges = []
        edge_labels = {}

        # ✅ Add nodes
        for node in self.metadata["nodes"]:
            G.add_node(node)

        # ✅ Add standard edges (Solid)
        for src, dst in self.metadata["edges"]:
            if dst == "__end__":
                dst = "END"
            G.add_edge(src, dst)
            solid_edges.append((src, dst))

        # ✅ Add conditional branching edges (Dashed)
        for node, conditions in self.metadata["conditional_edges"].items():
            for condition, target in conditions:
                G.add_edge(node, target)
                dashed_edges.append((node, target))
                edge_labels[(node, target)] = f"{condition}"

        # ✅ Layout for better separation
        plt.figure(figsize=(14, 8))
        pos = nx.spring_layout(G, seed=42)  # More natural positioning

        # **Draw Solid Edges (State Transitions)**
        nx.draw_networkx_edges(
            G, pos, edgelist=solid_edges,
            edge_color="black", width=2, alpha=0.8,
            arrows=True, arrowstyle="-|>", arrowsize=20,  # 🔥 Force arrows
            connectionstyle="arc3,rad=0.1"
        )

        # **Draw Dashed Conditional Edges (Branching Paths)**
        nx.draw_networkx_edges(
            G, pos, edgelist=dashed_edges,
            edge_color="red", style="dashed", width=2,
            arrows=True, arrowstyle="-|>", arrowsize=20,  # 🔥 Force arrows
            connectionstyle="arc3,rad=0.3"
        )

        # **Draw Nodes**
        nx.draw_networkx_nodes(G, pos, node_color="lightblue", node_size=2800, edgecolors="black")
        nx.draw_networkx_labels(G, pos, font_size=12, font_weight="bold")

        # **Draw Conditional Labels**
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color="red", font_size=10)

        # ✅ Highlight entry and finish points
        entry = self.metadata["entry_point"]
        finish = self.metadata["finish_point"]

        if entry:
            nx.draw_networkx_nodes(G, pos, nodelist=[entry], node_color="green", node_size=3000, edgecolors="black")  # Entry point
        if finish and finish in self.metadata["nodes"]:
            nx.draw_networkx_nodes(G, pos, nodelist=[finish], node_color="red", node_size=3000, edgecolors="black")  # Finish point

        # ✅ Save and display
        plt.title("State Graph Visualization", fontsize=14, fontweight="bold")
        plt.savefig(output_file, bbox_inches="tight")
        plt.show()
        print(f"Graph saved as {output_file}")

    def get_metadata(self):
        """Return the extracted metadata."""
        return self.metadata


In [22]:
graph_manager = StateGraphManager(agent.graph)

# Debug: Print before modifications
print("🔍 Before Modifications:", graph_manager.get_metadata())

# Insert a node between 'tool_node' and 'agent_node'
graph_manager.insert_node("new_mid_node", ("tool_node", "agent_node"))

# Insert a node into the START branch
graph_manager.insert_start_node("start_mid_node")

# Insert a node into the END branch
graph_manager.insert_end_node("final_check_node")

# Debug: Print after modifications
print("✅ After Modifications:", graph_manager.get_metadata())

# Force recompilation only when needed
graph_manager.ensure_compiled()


NameError: name 'agent' is not defined

In [ ]:
from IPython.display import Image, display

try:
    display(Image(agent.app.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
a_graph = StateGraphManager(agent.graph)

In [ ]:
# Initialize with a StateGraph
a_graph = StateGraphManager(agent.graph)

# Print extracted metadata
print(a_graph.get_metadata())

# Visualize the graph
a_graph.visualize()


In [ ]:
a

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

# Create a test graph
state_graph = StateGraph()
manager = StateGraphManager(state_graph)

# Step 1: Add Initial Nodes
print("➡️ Adding Initial Nodes and Edges")
state_graph.add_node("agent_node", lambda x: x)
state_graph.add_node("tool_node", lambda x: x)
state_graph.add_edge("agent_node", "tool_node")
state_graph.set_entry_point("agent_node")

print("✅ Initial Nodes:", state_graph.nodes.keys())
print("✅ Initial Edges:", state_graph.edges)

# Step 2: Insert a New Node (Before agent_node)
print("\n➡️ Inserting 'load_memories' Before 'agent_node'")
manager.insert_node("agent_node", "load_memories", lambda x: x)
print("✅ Updated Edges:", state_graph.edges)
manager.visualize()

# Step 3: Remove a Node
print("\n➡️ Removing 'tool_node'")
manager.remove_node("tool_node")
print("✅ Nodes After Removal:", state_graph.nodes.keys())
print("✅ Edges After Removal:", state_graph.edges)
manager.visualize()

# Step 4: Add Conditional Routing
print("\n➡️ Adding Conditional Branch at 'agent_node'")
def condition_fn(state):
    return "branch" if "use_branch" in state else "default"

manager.add_conditional_routing("agent_node", condi
